# 02 — ElasticNet 단일 회귀 (PP + scaling + HP joint Optuna)

선형 모델은 스케일/분포에 민감하므로 트리 4종과 달리 **PP + scaling + HP를 동시 탐색**한다.

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/enet/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 8축 joint search ([§9.1](strategy.md)) — `preprocess.run`을 매 trial마다 호출
- **Scaling**: 5종 categorical (`Standard / Robust / YeoJohnson / Quantile / Hybrid`) ([§9.2](strategy.md))
- **HPO**: 200 trial (joint이라 trial당 비용 큼), anchor 첫 trial enqueue ([§5.4](strategy.md), 출처: [4_output_이전자료/final/reg_only/enet/best_params.json](../../4_output_이전자료/final/reg_only/enet/best_params.json))
- **target transform**: `log1p` 고정 ON
- **leakage 방지**: `corr_keep_by` / `post_impute_corr_keep_by`는 `'std'` 고정 (`target_corr` 후보 제거 — KFold 안에서 train target으로 feature selection 하면 supervised leak)

## 모듈 의존성 ([strategy.md §13](strategy.md))

1. `3_modeling/modules/` 이관 완료 — `preprocess.run`이 cleaning + outlier(winsorize 0~99%)를 wrap
2. `models.py` — `enet_space`의 `l1_ratio` 0.5~0.95 (양 끝 제외, 본 노트북은 직접 trial loop)
3. `postprocess.py` — `Q25/Q75` + `zero_clip_space='log'`

본 노트북은 **trial loop를 직접 구성**해서 `preprocess.run` + scaling + ElasticNet을 매 trial마다 재실행한다.

## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# ── Google Drive 파일 ID (Colab 사용 시. 로컬은 무시됨) ──
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip (utils/, setup.py)
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip (CSV 4개)
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip (cleaning/outlier/scaling/...)
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # ★ modules.zip Drive ID (3_modeling/modules/) — 추후 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings, json, pickle, contextlib, io
warnings.filterwarnings('ignore')

import optuna
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, RobustScaler, PowerTransformer, QuantileTransformer

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from modules import preprocess, hpo   # noqa: E402  (preprocess.run이 cleaning+outlier wrap)
from scaling import HybridScaler   # 2_preprocessing/scaling.py
from meta_features import add_meta_features   # 2_preprocessing/meta_features.py — 2026-05-09 결정 반영

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

## 2. 실험 설정

In [ ]:
MODEL_NAME = 'enet'
EXP_ID     = f'reg-{MODEL_NAME}-002'
EXP_MEMO   = 'PP+scaling+y_transform+HP joint Optuna (1차 OOF=0.005563)'
USER       = 'jh'

N_TRIALS    = 1   # joint이라 비용 큼
N_FOLDS     = 5
N_STARTUP_TRIALS = 1
N_JOBS      = 14    # enet 단독 (strategy.md §12)
TIMEOUT_SEC = None  # ★ Colab 타임아웃 대비. 초 단위, None=무제한

CLIP_Y_EXTREME   = True

OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# anchor (strategy.md §5.4, 1차 OOF=0.005563)
# joint 탐색 — PP/scaling/target_transform 키도 anchor에 포함
ENET_ANCHOR = {
    # PP 8축
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'add_indicator':              True,
    'indicator_threshold':        0.10,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.97,
    # X scaling
    'scaling':                    'RobustScaler',
    # y target transform (★ 1차 best 검증된 변환)
    'target_transform':           'log1p',
    # enet HP
    'alpha':                      5.377e-06,
    'l1_ratio':                   0.885,
    'max_iter':                   15000,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}')
print(f'TIMEOUT_SEC={TIMEOUT_SEC} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')


## 3. 데이터 로드 + target clip + transform

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# target_transform은 trial 안에서 fold별 fit (글로벌 함수 제거)
print('[target transform] Optuna가 trial별 자동 탐색: none / log1p / yeo-johnson / quantile')


## 4. Joint trial 헬퍼 (scaling factory + KFold)

[strategy_common.md §6](../strategy_common.md): `KFold(unit_ids)` → die mask. 모든 trial이 같은 fold split을 공유.

PP는 [`preprocess.run`](../modules/preprocess.py) (Stage 0 + cleaning + winsorize wrap) 을 매 trial마다 호출.

In [ ]:
# unit-level KFold (모든 trial 공유)
unit_ids_train = ys_input['train'][KEY_COL].unique()
_kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = []
for tr_idx, vl_idx in _kf.split(unit_ids_train):
    FOLDS.append((unit_ids_train[tr_idx], unit_ids_train[vl_idx]))
print(f'[KFold] {N_FOLDS} folds, unit 단위 분할 완료')


def make_scaler(name):
    if name == 'StandardScaler':
        return StandardScaler()
    if name == 'RobustScaler':
        return RobustScaler()
    if name == 'YeoJohnson':
        return PowerTransformer(method='yeo-johnson', standardize=True)
    if name == 'Quantile':
        return QuantileTransformer(output_distribution='normal', random_state=SEED)
    if name == 'Hybrid':
        return HybridScaler(skew_threshold=10.0)
    raise ValueError(f'Unknown scaling: {name!r}')


def make_target_transformer(name, y_train_arr):
    """fold별 train fold y에 fit한 (forward_fn, inverse_fn) 반환.

    선형 모델은 inverse 후 음수 가능 → 모든 inverse는 np.clip(0, None) 적용.
    leakage 방지 위해 transformer는 반드시 train fold y에만 fit.

    후보 (strategy_common.md §3):
    - 'none'        : identity
    - 'log1p'       : np.log1p / np.expm1 (1차 enet best 검증)
    - 'yeo-johnson' : PowerTransformer (lambda 자동 추정)
    - 'quantile'    : QuantileTransformer (rank → normal)
    """
    if name == 'none':
        return (
            lambda y: np.asarray(y, dtype=float),
            lambda y: np.clip(np.asarray(y, dtype=float), 0.0, None),
        )
    if name == 'log1p':
        return (
            lambda y: np.log1p(np.asarray(y, dtype=float)),
            lambda y: np.clip(np.expm1(np.asarray(y, dtype=float)), 0.0, None),
        )
    if name == 'yeo-johnson':
        pt = PowerTransformer(method='yeo-johnson', standardize=False)
        pt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: pt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(pt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    if name == 'quantile':
        n_q = min(1000, len(y_train_arr))
        qt = QuantileTransformer(output_distribution='normal', n_quantiles=n_q, random_state=SEED)
        qt.fit(np.asarray(y_train_arr, dtype=float).reshape(-1, 1))
        return (
            lambda y: qt.transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(),
            lambda y: np.clip(qt.inverse_transform(np.asarray(y, dtype=float).reshape(-1, 1)).ravel(), 0.0, None),
        )
    raise ValueError(f'Unknown target_transform: {name!r}')


def run_pp_silent(pp_params):
    """preprocess.run 호출. corr_keep_by/post_impute_corr_keep_by는 'std' 고정 (leakage 방지)."""
    pp_full = dict(pp_params,
                   corr_keep_by='std',
                   post_impute_corr_keep_by='std')
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
    return out


print('[helper] make_scaler / make_target_transformer / run_pp_silent 정의 완료')


## 5. Optuna joint HPO (PP + scaling + enet HP)

In [ ]:
y_train_unit_df = ys_input['train']
y_train_unit    = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]


def _broadcast_y_to_die(xs_split, y_unit_series):
    return xs_split[KEY_COL].map(y_unit_series).values.astype(float)


def _aggregate_die_to_unit_mean(xs_split, die_pred):
    df = pd.DataFrame({KEY_COL: xs_split[KEY_COL].values, 'pred': die_pred})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    # ── PP 8축 (strategy.md §9.1) ──
    pp_params = {
        'missing_threshold':          trial.suggest_float('missing_threshold', 0.30, 0.90),
        'corr_threshold':             trial.suggest_float('corr_threshold', 0.88, 0.98),
        'add_indicator':              trial.suggest_categorical('add_indicator', [True, False]),
        'indicator_threshold':        trial.suggest_float('indicator_threshold', 0.05, 0.20),
        'spatial_max_dist':           trial.suggest_float('spatial_max_dist', 1.0, 6.0),
        'post_impute_corr_threshold': trial.suggest_float('post_impute_corr_threshold', 0.96, 0.99),
        # corr_keep_by / post_impute_corr_keep_by: 'std' 고정 (helper에서 강제)
    }
    # ── X scaling 5종 ──
    scaling_name = trial.suggest_categorical(
        'scaling', ['StandardScaler', 'RobustScaler', 'YeoJohnson', 'Quantile', 'Hybrid']
    )
    # ── y target_transform 4종 (strategy_common §3) ──
    target_transform_name = trial.suggest_categorical(
        'target_transform', ['none', 'log1p', 'yeo-johnson', 'quantile']
    )
    # ── enet HP (strategy.md §7.4) ──
    enet_hp = dict(
        alpha=trial.suggest_float('alpha', 1e-7, 1e-4, log=True),
        l1_ratio=trial.suggest_float('l1_ratio', 0.50, 0.95),
        max_iter=trial.suggest_int('max_iter', 8000, 20000, step=1000),
        tol=1e-6,
        selection='random',
        precompute=True,
        random_state=SEED,
    )

    # ── 1) PP (preprocess.run wrap) ──
    try:
        out = run_pp_silent(pp_params)
    except Exception as e:
        raise optuna.exceptions.TrialPruned(f'PP failed: {e}')
    xs_train_c      = out['xs_train']
    feat_cols_clean = out['feat_cols']

    # ── 1.5) 메타피처 추가 (2026-05-09 결정: enet=position OHE, die_xy 제외) ──
    feat_cols_clean = add_meta_features(
        xs_train_c, None, None, feat_cols_clean,
        position_mode='ohe', use_die_xy=False, verbose=False,
        use_loc_x_ohe=True,
        loc_x_required=("X1073",),
        loc_x_optional=("X1059", "X1075", "X1076", "X1077"),
    )

    y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

    n_tr = len(xs_train_c)
    oof = np.full(n_tr, np.nan)

    # ── 2) fold 단위 학습 (target transformer는 fold별 fit) ──
    for tr_units, vl_units in FOLDS:
        tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
        vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

        X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
        X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values

        # target transformer는 fold별 train fold y에만 fit (leakage 방지)
        y_tr_orig = y_die_orig[tr_mask]
        forward_fn, inverse_fn = make_target_transformer(target_transform_name, y_tr_orig)
        y_tr = forward_fn(y_tr_orig)

        scaler = make_scaler(scaling_name)
        X_tr_s = scaler.fit_transform(X_tr)
        X_vl_s = scaler.transform(X_vl)

        try:
            model = ElasticNet(**enet_hp)
            model.fit(X_tr_s, y_tr)
            pred_t = model.predict(X_vl_s)
        except Exception as e:
            raise optuna.exceptions.TrialPruned(f'enet fit/predict failed: {e}')
        oof[vl_mask] = inverse_fn(pred_t)

    if np.isnan(oof).any():
        raise RuntimeError('OOF has NaN — fold coverage bug')

    # unit-level RMSE (mean 집계, postprocess 이전)
    unit_pred = _aggregate_die_to_unit_mean(xs_train_c, oof)
    aligned   = unit_pred.set_index(KEY_COL)['pred'].loc[y_train_unit.index]
    train_rmse = float(np.sqrt(np.mean((aligned.values - y_train_unit.values) ** 2)))
    trial.set_user_attr('train_rmse', train_rmse)
    trial.set_user_attr('n_features_after_pp', len(feat_cols_clean))
    return train_rmse


# anchor 첫 trial enqueue
study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=False,
    sampler=optuna.samplers.TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=10),
)
study.set_user_attr('exp_id', EXP_ID)
study.set_user_attr('exp_memo', EXP_MEMO)
study.set_user_attr('user', USER)
study.set_user_attr('anchor', ENET_ANCHOR)
study.enqueue_trial(ENET_ANCHOR)

# ★ Colab 타임아웃 대비 — TIMEOUT_SEC 초과 시 graceful 종료 (None=무제한)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

best_params = dict(study.best_trial.params)
best_value  = float(study.best_value)
print(f'\n[HPO 완료] best OOF RMSE = {best_value:.6f}')
print(f'[검증] trial 0 params (anchor 키만): { {k: study.trials[0].params.get(k) for k in ENET_ANCHOR if k in study.trials[0].params} }')
print(f'best_params = {best_params}')

## 6. Best trial 재학습 (전체 fold OOF + val/test)

In [ ]:
# best_params에서 PP / scaling / target_transform / enet HP 분리
PP_KEYS = ('missing_threshold', 'corr_threshold', 'add_indicator',
           'indicator_threshold', 'spatial_max_dist', 'post_impute_corr_threshold')
best_pp               = {k: best_params[k] for k in PP_KEYS if k in best_params}
best_scaling          = best_params.get('scaling', 'RobustScaler')
best_target_transform = best_params.get('target_transform', 'log1p')
best_enet             = {k: best_params[k] for k in ('alpha', 'l1_ratio', 'max_iter')}
best_enet.update(tol=1e-6, selection='random', precompute=True, random_state=SEED)

# best 조합으로 PP 한 번 (refit은 print 보존)
pp_full = dict(best_pp, corr_keep_by='std', post_impute_corr_keep_by='std')
out = preprocess.run(xs.copy(), ys_input, feat_cols, xs_dict, params=pp_full)
xs_train_c      = out['xs_train']
xs_val_c        = out['xs_val']
xs_test_c       = out['xs_test']
feat_cols_clean = out['feat_cols']

# ── 메타피처 추가 (2026-05-09 결정: enet=position OHE, die_xy 제외) ──
feat_cols_clean = add_meta_features(
    xs_train_c, xs_val_c, xs_test_c, feat_cols_clean,
    position_mode='ohe', use_die_xy=False,
    use_loc_x_ohe=True,
    loc_x_required=("X1073",),
    loc_x_optional=("X1059", "X1075", "X1076", "X1077"),
)

y_die_orig = _broadcast_y_to_die(xs_train_c, y_train_unit)

n_tr, n_vl, n_te = len(xs_train_c), len(xs_val_c), len(xs_test_c)
oof_pred  = np.full(n_tr, np.nan)
val_pred  = np.zeros(n_vl)
test_pred = np.zeros(n_te)

fold_models  = []
fold_scalers = []

for i, (tr_units, vl_units) in enumerate(FOLDS):
    tr_mask = xs_train_c[KEY_COL].isin(set(tr_units)).values
    vl_mask = xs_train_c[KEY_COL].isin(set(vl_units)).values

    X_tr = xs_train_c.loc[tr_mask, feat_cols_clean].values
    X_vl = xs_train_c.loc[vl_mask, feat_cols_clean].values
    X_v  = xs_val_c[feat_cols_clean].values
    X_te = xs_test_c[feat_cols_clean].values

    # ── fold별 target transformer fit (objective와 동일 패턴) ──
    y_tr_orig = y_die_orig[tr_mask]
    forward_fn, inverse_fn = make_target_transformer(best_target_transform, y_tr_orig)
    y_tr = forward_fn(y_tr_orig)

    scaler = make_scaler(best_scaling)
    X_tr_s = scaler.fit_transform(X_tr)
    X_vl_s = scaler.transform(X_vl)
    X_v_s  = scaler.transform(X_v)
    X_te_s = scaler.transform(X_te)

    model = ElasticNet(**best_enet)
    model.fit(X_tr_s, y_tr)

    oof_pred[vl_mask] = inverse_fn(model.predict(X_vl_s))
    val_pred  += inverse_fn(model.predict(X_v_s))  / N_FOLDS
    test_pred += inverse_fn(model.predict(X_te_s)) / N_FOLDS

    fold_models.append(model)
    fold_scalers.append(scaler)
    print(f'[refit fold {i+1}/{N_FOLDS}] tr_units={len(tr_units)}, vl_units={len(vl_units)}')

# unit-level RMSE (mean 집계)
oof_unit  = _aggregate_die_to_unit_mean(xs_train_c, oof_pred)
val_unit  = _aggregate_die_to_unit_mean(xs_val_c,   val_pred)
test_unit = _aggregate_die_to_unit_mean(xs_test_c,  test_pred)

y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

oof_rmse  = float(np.sqrt(np.mean((oof_unit.set_index(KEY_COL)['pred'].loc[y_train_unit.index].values - y_train_unit.values)**2)))
val_rmse  = float(np.sqrt(np.mean((val_unit.set_index(KEY_COL)['pred'].loc[y_val_true.index].values  - y_val_true.values)**2)))
test_rmse = float(np.sqrt(np.mean((test_unit.set_index(KEY_COL)['pred'].loc[y_test_true.index].values - y_test_true.values)**2)))

print(f'\n[Refit 완료] (postprocess 이전)')
print(f'  best target_transform = {best_target_transform}')
print(f'  best scaling          = {best_scaling}')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 후처리 매트릭스 + 산출물 저장

In [ ]:
# refit_result dict — hpo.save_artifacts 호환 형식으로 구성
refit_result = {
    'oof_pred_die':         oof_pred,
    'val_pred_die':         val_pred,
    'test_pred_die':        test_pred,
    'oof_pi':  None, 'val_pi':  None, 'test_pi':  None,
    'oof_mu':  None, 'val_mu':  None, 'test_mu':  None,
    'oof_pred_unit':        oof_unit,
    'val_pred_unit':        val_unit,
    'test_pred_unit':       test_unit,
    'fold_models':          fold_models,
    'fold_scalers':         fold_scalers,
    'best_params_resolved': {**best_pp,
                             'scaling':          best_scaling,
                             'target_transform': best_target_transform,
                             **best_enet},
    'model_name':           MODEL_NAME,
}

# zero_clip은 학습 공간(target_transform)과 일치시킴:
# 'log1p' → log space에서 임계값 비교 / 그 외(none/yeo/quantile) → original space
zero_clip_log_space = (best_target_transform == 'log1p')

POSTPROCESS_CONFIG = {
    'agg_methods':         ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
    'zero_clip_range':     (0.001, 0.015),
    'zero_clip_step':      0.001,
    'zero_clip_log_space': zero_clip_log_space,
    'use_pi_threshold':    False,
}

study_meta_for_save = {
    'exp_id':                EXP_ID,
    'exp_memo':              EXP_MEMO,
    'user':                  USER,
    'model_name':            MODEL_NAME,
    'best_target_transform': best_target_transform,   # ★ 신규
    'best_scaling':          best_scaling,
    'best_pp':               best_pp,
    'clip_y_extreme':        CLIP_Y_EXTREME,
    'n_trials':              N_TRIALS,
    'n_folds':               N_FOLDS,
    'n_jobs':                N_JOBS,
    'timeout_sec':           TIMEOUT_SEC,             # ★ 신규
    'seed_kfold':            SEED,
    'anchor':                ENET_ANCHOR,
    'hpo_best_value':        best_value,
}

hpo.save_artifacts(
    refit_result=refit_result,
    xs_train=xs_train_c, xs_val=xs_val_c, xs_test=xs_test_c,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass
